Реализация тематического моделирования через sklearn

В качестве корпуса взято 7 рандомных текстов из корпуса "100 великих"

https://drive.google.com/drive/folders/1sfjFpEZpvVnUGrclvlAeMwu_NO1h6KyW?usp=share_link

In [1]:
import codecs
import os

In [2]:
from sklearn.decomposition import LatentDirichletAllocation

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

In [4]:
from tqdm import notebook

In [5]:
import spacy

In [6]:
nlp = spacy.load('ru_core_news_sm')

In [7]:
PATH = r'cor'
NUM_TOPICS = 7
N_FIRST = 20 # сколько первых слов на тему выводить
MIN_DF = 5 # минимальное число вхождений слова в документ, чтобы мы его учитывали (включали в вектор документа)
MAX_DF = 0.90 # в каком проценте документов должно присутствовать слово, чтобы оно не рассматривалось
STW_PATH = 'swl.txt' # путь к документу со стоп-словами

In [8]:
with codecs.open(STW_PATH, encoding = 'utf-8') as f:
    stw_list = f.read().split()[1:] # тк первый символ - что-то странное: \ufeffа 

In [9]:
file_names = os.listdir(PATH)
file_names

['Новый текстовый документ (2).txt',
 'Новый текстовый документ (3).txt',
 'Новый текстовый документ (4).txt',
 'Новый текстовый документ (5).txt',
 'Новый текстовый документ (6).txt',
 'Новый текстовый документ (7).txt',
 'Новый текстовый документ.txt']

In [ ]:
for filename in file_names:
    file_text = ''
    with open(os.path.join(PATH, filename), 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in notebook.tqdm(lines):
            doc = nlp(line.strip())
            clear = ''
            for token in doc:
                if token.text.isalpha():
                    clear += f'{token.lemma_} '
            clear = clear.strip()
            file_text += f'{clear}\n'
    with open(filename[:-4]+'_annot.txt', 'w', encoding='utf-8') as f:
        f.write(file_text)

Читаем файлы в переменную=корпус, которую потом подадим на вход ТМ.
ТМ принимает ""An iterable which generates either str, unicode or file objects.""

In [11]:
file_names = os.listdir(r'annot') 
data = []
for name in file_names:
    if name.endswith(".txt"):
        with codecs.open(r'annot' + "\\" + name, encoding = 'utf-8') as f:
            data.append(f.read())
            # each document should contain lemmatized words separated by spaces

In [12]:
count_vectorizer = CountVectorizer(min_df=MIN_DF, max_df=MAX_DF, stop_words = stw_list)
data_count_vectorized = count_vectorizer.fit_transform(data)

C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ак', 'бляха', 'богу', 'бу', 'буль', 'вашему', 'вообще', 'всякому', 'го', 'другому', 'ей', 'елки', 'иному', 'кей', 'кис', 'кэй', 'ля', 'мое', 'моему', 'муха', 'нашему', 'нить', 'палки', 'паф', 'пиф', 'пли', 'своему', 'твоему', 'тик', 'тра', 'тс', 'ту', 'чегой', 'черт', 'чик', 'чтой', 'яй'] not in stop_words.
  warnings.warn(


In [13]:
lda_model = LatentDirichletAllocation(n_components=NUM_TOPICS, max_iter=20, learning_method='batch', batch_size=512, random_state = 2)
lda_model.fit_transform(data_count_vectorized)

array([[6.83067440e-06, 3.38927572e-01, 4.19522091e-03, 6.83001542e-06,
        6.80188496e-06, 6.80188496e-06, 6.56849943e-01],
       [6.28691724e-02, 1.10119091e-05, 5.28685906e-02, 2.78909714e-02,
        1.09698711e-05, 1.09698711e-05, 8.56338314e-01],
       [8.16999535e-06, 8.17161684e-06, 8.16978141e-06, 9.99951052e-01,
        8.13460712e-06, 8.13460712e-06, 8.16726872e-06],
       [9.37571976e-01, 3.69994633e-02, 7.71013390e-06, 2.53977869e-02,
        7.67612044e-06, 7.67612044e-06, 7.71123103e-06],
       [9.58460966e-01, 7.94836740e-03, 6.95658283e-05, 1.02838544e-05,
        1.02408884e-05, 1.02408884e-05, 3.34903355e-02],
       [5.97604221e-06, 9.99964196e-01, 5.97160405e-06, 5.97458596e-06,
        5.95249585e-06, 5.95249585e-06, 5.97634048e-06],
       [7.03454589e-06, 7.03355216e-06, 9.99957825e-01, 7.03775740e-06,
        7.01425037e-06, 7.01425037e-06, 7.04073343e-06]])

В ячейке ниже был код, использующий устаревшую версию либы и обращавшийся к vectorizer.get_feature_names(). 

Замена на vectorizer.get_feature_names_out() ведет к корректной работе.

In [14]:
# print function
all_topics = []
def print_topics(model, vectorizer, top_n=N_FIRST):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        all_topics.append([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-top_n - 1:-1]])
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-top_n - 1:-1]])

In [15]:
print_topics(lda_model, count_vectorizer)

Topic 0:
[('портрет', 905.7878740045334), ('полотно', 401.0683095852225), ('композиция', 376.0322423852544), ('зритель', 353.0630260063837), ('творчество', 350.8221085643205), ('мастер', 341.95769952422154), ('художественный', 317.31877830782315), ('выставка', 291.02887054664757), ('рисунок', 249.33616903687724), ('живописный', 212.0556628815679), ('мастерская', 201.94200366073449), ('христос', 200.8076375413893), ('сюжет', 195.26211593927982), ('натура', 169.76315032892856), ('кисть', 160.0657946805428), ('творческий', 150.94135743583507), ('церковь', 149.29674667908336), ('ученик', 118.08943886098554), ('леонардо', 116.80242888080869), ('голубой', 114.90114331481917)]
Topic 1:
[('заговор', 548.1368878834859), ('переворот', 286.77297874295573), ('министр', 279.2933254156225), ('император', 276.12148347402047), ('политический', 273.74265379346076), ('приказ', 239.80502568156044), ('гитлер', 227.6407418218539), ('убийство', 201.55628754935623), ('князь', 188.9550633832472), ('герцог', 1

Ползунок справа (λ) отвечает за специфичность слов к теме. Чем ближе к 0, тем более специфичные слова. Чем ближе к 1 - тем выше вероятность, что они встречаются еще где-то.

In [16]:
# визуализация
import pyLDAvis.lda_model

#import sklearn.pyLDAvis
 
pyLDAvis.enable_notebook()
panel = pyLDAvis.lda_model.prepare(lda_model, data_count_vectorized, count_vectorizer, mds='tsne')
panel

C:\Users\Андрей\AppData\Roaming\Python\Python310\site-packages\pandas\core\dtypes\cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


PreparedData(topic_coordinates=               x          y  topics  cluster       Freq
topic                                                  
1      -2.262535  -8.154224       1        1  24.835739
0      -5.869885  52.353642       2        1  24.618542
6      51.933823 -35.453449       3        1  19.776148
2     -56.653294  19.024050       4        1  16.452626
3      48.364159  25.135439       5        1  14.315389
4     -52.801975 -41.516090       6        1   0.000778
5       1.276050 -68.668465       7        1   0.000778, topic_info=                 Term        Freq       Total Category  logprob  loglift
2222          портрет  905.000000  905.000000  Default  30.0000  30.0000
3168              ток  584.000000  584.000000  Default  29.0000  29.0000
599         двигатель  491.000000  491.000000  Default  28.0000  28.0000
800           заговор  544.000000  544.000000  Default  27.0000  27.0000
1082           колесо  388.000000  388.000000  Default  26.0000  26.0000
...               ...         ...         ...      ...      ...      ...
3242         увенчать    0.000277    7.562336   Topic7  -8.1917   1.5488
2514    проповедовать    0.000277   11.476767   Topic7  -8.1917   1.1317
2861              сим    0.000277    6.583723   Topic7  -8.1917   1.6874
1157  кратковременный    0.000277    7.552753   Topic7  -8.1917   1.5501
1142          коротко    0.000277    9.538409   Topic7  -8.1917   1.3167

[409 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
0         1  0.794594      iii
0         2  0.118919      iii
0         3  0.021622      iii
0         4  0.064865      iii
23        1  0.282547  адмирал
...     ...       ...      ...
3600      5  0.465047  ядерный
3602      1  0.015112     яйцо
3602      3  0.891624     яйцо
3602      4  0.060449     яйцо
3602      5  0.015112     яйцо

[1157 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 1, 7, 3, 4, 5, 6])

In [17]:
lda_model.perplexity(data_count_vectorized)

1611.6195662222042

In [18]:
from gensim.models import CoherenceModel
import gensim.corpora as corpora

def get_Cv(model, df_columnm):
    topics = model.components_

    n_top_words = 20
    texts = [[word for word in doc.split()] for doc in df_columnm]

    # Create a gensim dictionary from the word count matrix
    dictionary = corpora.Dictionary(texts)
    
    # Create a gensim corpus from the word count matrix
    corpus = [dictionary.doc2bow(text) for text in texts]

    feature_names = [dictionary[i] for i in range(len(dictionary))]

    # Get the top words for each topic from the components_ attribute
    top_words = []
    for topic in topics:
        top_words.append([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])

    coherence_model = CoherenceModel(topics=top_words, texts=texts, dictionary=dictionary, coherence='c_v')
    coherence = coherence_model.get_coherence()
    return coherence

In [ ]:
'Coherence value is ', get_Cv(lda_model, data)